# AStats: Agentic-AI Statistical Workflow Prototype

## Overview
This prototype demonstrates the **AStats** vision: using **Agentic-AI** (LLMs as orchestrators) to guide statistical practitioners through complex workflows. 

### Key Capabilities:
1. **Auto-Discovery**: Automatic loading and schema inference of any CSV.
2. **Structure-Aware Profiling**: Detecting grouping columns and **repeated measures** (a common pitfall for traditional agents).
3. **Assumption-Driven Selection**: Rule-based logic (Shapiro-Wilk, Levene) ensures correct test choices.
4. **Agentic Chain-of-Thought**: An LLM agent (transformers/FLAN-T5) plans steps and justifies choices in natural language.
5. **Reproducibility**: Every decision is logged in a structured format.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from src.eda import EDA
from src.profiler import Profiler
from src.test_selector import TestSelector
from src.executor import Executor
from src.agent import AStatsAgent
from src.logger import logger
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Case Study 1: Sleepstudy (Repeated Measures)
We use a synthetic `sleepstudy` dataset where multiple subjects are measured over 10 days. The agent should detect the `Subject` ID and recommend a repeated measures test.

In [ ]:
csv_path = "data/sleepstudy.csv"
df = pd.read_csv(csv_path)
print(df.head())

sns.lineplot(data=df, x='Days', y='Reaction', hue='Subject')
plt.title("Sleepstudy: Reaction vs Days (By Subject)")
plt.show()

### Run the Agentic Pipeline

In [ ]:
# 1. Initialize Agent
agent = AStatsAgent()

# 2. Profiling & Structure Detection
profiler = Profiler(df)
vtypes = profiler.classify_variables()
id_candidates = profiler.detect_repeated_measures()

id_col = id_candidates[0]['column'] if id_candidates else None

print(f"Detected Variable Types: {vtypes}")
print(f"Primary ID/Group for Repeated Measures: {id_col}")

# 3. Test Selection (Heuristic comparison: Reaction vs Days)
selector = TestSelector(df, vtypes)
test_info = selector.select_test("Reaction", "Days", id_col=id_col)

# 4. Agentic Logic (LLM Reflection)
agent.justify_test(test_info['test'], "Repeated measures detected on Subject")

# 5. Execute
executor = Executor(df)
results = executor.run_test(test_info)
print(f"Final Statistical Result: {results}")

## Conclusion & Reproducibility Log
All steps taken by the agent are recorded in the structured workflow log for audit and scientific reproducibility.

In [ ]:
logger.save_log("notebook_analysis_record.json")